# Foundational SAT Skill Gap Prediction

**Synthetic data only.** No real student data.

## Flow (response-based)
1. Synthetic MCQ **item bank** (3 items × 72 skills)
2. Students attempt items for ~60% of skills → **response matrix R**
3. Aggregate: **S = normalize(R · Q)** → skill mastery %
4. Untested skills = no items attempted
5. Cosine similarity on observed skills → predict gaps

In [ ]:
import numpy as np
import pandas as pd
from lib import run_pipeline, analyze_student, cosine_similarity_observed, item_cosine_similarity
from export_frontend import export_dashboard
from items import ITEMS, write_item_bank

## 1. Item bank & response matrix R

Each item maps to one skill. Matrix **R** has shape (students × items); entry is 1 if correct, 0 if wrong, NaN if not attempted.

In [ ]:
result = run_pipeline(seed=42)
print(f"Items: {len(result.item_ids)}, Students: {len(result.student_ids)}")
print(f"Response matrix R shape: {result.response_matrix.shape}")
print(f"Skill-item matrix Q shape: {result.skill_item_matrix.shape}")
result.responses_df.head()

## 2. Aggregate to skill matrix S

$$S_{i,j} = 100 \times \frac{\sum_k R_{i,k} Q_{k,j}}{\sum_k Q_{k,j} \cdot \mathbb{1}[R_{i,k} \text{ attempted}]}$$

Implemented as group-by: % correct items per (student, skill).

In [ ]:
S = result.scores
M = result.mask
print(f"Skill matrix S: {S.shape}")
print(f"Tested skills per student: {M.sum(axis=1).mean():.1f} avg")
print(f"Untested fraction: {(~M).mean():.1%}")

## 3. Sample student work (auditable)

Every displayed mastery score equals % correct from these item attempts.

In [ ]:
target_idx = result.default_student_idx
sid = result.student_ids[target_idx]
work = result.responses_df[result.responses_df.student_id == sid]
print(f"Student {sid}: {len(work)} items across {M[target_idx].sum()} skills")
work[["skill_name", "item_id", "chosen", "correct_choice", "is_correct"]].head(12)

## 4. Cosine similarity (skill vectors & item vectors)

**Skill layer** (used for predictions):
$$\text{sim}(u,v) = \frac{u_\Omega \cdot v_\Omega}{\|u_\Omega\| \|v_\Omega\|}$$

**Item layer** (demo): same formula on shared attempted items in R.

In [ ]:
analysis = analyze_student(result, target_idx)
peer_idx = 1
shared = M[target_idx] & M[peer_idx]
skill_sim = cosine_similarity_observed(S[target_idx], S[peer_idx], shared)
item_sim = item_cosine_similarity(result.response_matrix, target_idx, peer_idx)
print(f"Skill-vector similarity: {skill_sim:.4f}")
print(f"Item-vector similarity: {item_sim:.4f}")

## 5. Predictions & priority ranking

In [ ]:
recs = pd.DataFrame(analysis["recommendations"])
print(analysis["summary"]["interpretation"])
recs.head(10)[["rank", "skill_name", "category", "level", "predicted_mastery", "priority_score", "reason"]]

In [ ]:
from lib import save_figures
save_figures(result, target_idx, analysis)
export_dashboard()
print("Figures + dashboard.json exported")

## Assignment requirements

1. **Linear algebra:** R, Q, S matrices; dot products; norms; cosine similarity; weighted completion.
2. **Real problem:** Prioritize untested foundational skills when full coverage is impossible.
3. **Verifiable artifacts:** Item bank, response CSV, skill scores derived from work, visualizations, dashboard.